# Modeling of Flood-Related Infrastructure Stress and Institutional Response in New York City

**Mario Alberto Ponce-Pacheco**  
Center for Urban Science and Progress (CUSP) — New York University  

---

## I. Background

Urban flooding poses a growing threat to infrastructure, public safety, and social equity in coastal cities. New York City is particularly exposed: its aging combined sewer system, tidal coastline, densely built environment, and rapidly intensifying precipitation create compounding stressors that vary markedly across neighborhoods. The devastation wrought by Hurricane Sandy in 2012 illustrated how a single hydrometeorological event could cascade into prolonged infrastructure dysfunction — flooded streets, disrupted mobility, overwhelmed emergency response — with impacts concentrated disproportionately in low-income and transit-dependent communities. Despite substantial post-Sandy investment in resilience infrastructure, routine flooding from nuisance storms, pipe surcharges, and tidal inundation continues to generate thousands of annual service requests to the city's 311 system, each one an administrative signal of localized infrastructure stress.

A growing body of literature connects precipitation extremes, tidal forcing, and drainage network characteristics to flood occurrence (Rosenzweig et al., 2019; Depietri & McPhearson, 2018), but comparatively little work has modeled the *institutional response* side of flooding — how quickly the city acknowledges, investigates, and resolves reported events, and whether that response capacity varies systematically by geography, political administration, or community vulnerability. Machine learning approaches applied to administrative data offer a complementary analytical lens: rather than simulating physical flood processes from first principles, they characterize the data-generating process behind observed complaints and service closures, revealing systematic patterns and disparities that physically-based models may miss.

Prior work on environmental justice and 311 data has demonstrated that complaint rates and service response times differ by race, income, and political geography in U.S. cities (Kontokosta & Hong, 2021; Minkoff, 2016; O'Brien et al., 2017). This project extends that tradition to the flood domain, situating itself at the intersection of urban climatology, network science, and computational social science. Drawing on 311 administrative records spanning four mayoral administrations (Bloomberg through Mamdani, 2002–2026), it models flood infrastructure stress as a multi-stage prediction problem with explicit fairness auditing.

## II. Problem Statement

This study addresses the following central question: *Can we predict, from structural and hydrometeorological features of a street segment and event window, whether a flood-related infrastructure complaint will occur, how intense the reported stress will be, and how quickly the city will resolve it?*

The question is decomposed into three interdependent modeling tasks. First, *occurrence*: given a spatiotemporal event window defined by a precipitation or tidal event and a street segment, does a 311 flood complaint arise? Second, *intensity*: conditional on complaint occurrence, what is the reported intensity of the event, measured by complaint volume per segment per event window? Third, *resolution*: does the complaint get formally closed, and if so, how many hours does closure take? Each task corresponds to a distinct stage of the infrastructure stress cycle — demand generation, demand magnitude, and institutional response capacity.

Beyond prediction accuracy, the study examines whether outcomes exhibit systematic disparities across geographies and political periods. Specifically: Do boroughs with lower median household income or higher concentrations of renters experience longer resolution times or higher false-negative rates in occurrence prediction? Do error rates shift across mayoral administrations in ways that suggest policy-driven changes in service delivery? And which feature classes — hydrometeorological forcing, network topology, drainage proximity, or socioeconomic context — contribute most to predictive performance?

The answers to these questions have direct practical implications. A reliable occurrence model could guide proactive drain inspection and maintenance prioritization before storms arrive. A resolution-time model could flag segments likely to experience protracted service delays, enabling targeted crew deployment. An equity analysis of model errors could identify underserved areas where current resource allocation patterns perpetuate differential flood vulnerability — a key concern for equitable resilience planning in NYC and comparable coastal cities facing intensifying climate stressors.

## III. Methodology

The analytical pipeline consists of six integrated stages.

**Data integration.** A master analytical table is constructed at the street segment × storm event granularity. 311 flood complaints (2002–2026) are spatially and temporally joined to: (i) street network data from OpenStreetMap, including travel time, edge betweenness, and network criticality metrics computed via NetworkX; (ii) FEMA Flood Insurance Rate Map designations, including Special Flood Hazard Area overlap by segment; (iii) drainage infrastructure from the NYC Department of Environmental Protection (catch basins, outfalls); (iv) NOAA tidal gauge observations from stations at Battery Park, Kings Point, and Sandy Hook; (v) precipitation records from ASOS ground stations and NEXRAD radar polygons; and (vi) American Community Survey (ACS) socioeconomic indicators at the census tract level.

**Labeling and balancing.** Events are defined by merging spatially proximate 311 complaints into contiguous temporal windows. Occurrence is a binary target (complaint present vs. matched non-flood control). The occurrence target is heavily imbalanced; stratified subsampling equalizes positive and negative classes while preserving mayoral administration representation. Intensity is a continuous count variable (log1p-transformed). Resolution time is the hours elapsed from complaint open to close, also log1p-transformed, with a binary closure indicator.

**Clustering and anomaly detection.** K-Means, agglomerative, and Gaussian Mixture Models are applied across three feature subspaces: physical forcing (precipitation, tide, terrain), infrastructure-network topology, and social-governance context. Isolation Forest identifies atypical events.

**Supervised learning with temporal validation.** Three ML tasks are executed with a forward-looking temporal split: train on Bloomberg-era data (pre-2014), validate on de Blasio-era data (2014–2021), test on Adams/Mamdani data (2022–present). A baseline stratified 75/15/10 split is computed for comparison. Classification models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting. Regression models: Ridge, Poisson, Tweedie, Random Forest Regressor, and Gradient Boosting Regressor.

**Fairness auditing.** Bias diagnostics partition residuals and error rates by borough, mayoral administration, and poverty tercile to detect differential model performance.

## IV. Results

**Occurrence classification.** The Random Forest model achieves the strongest test-set performance under the stratified split: ROC-AUC = 0.892, F1 = 0.817, precision = 0.798, recall = 0.838. Gradient Boosting is a close second (ROC-AUC = 0.889, F1 = 0.811). Logistic Regression underperforms substantially (ROC-AUC = 0.764), confirming the non-linear nature of flood occurrence patterns. The top two predictors by Gini importance are street network travel time (0.117) and FEMA Special Flood Hazard Area overlap in feet (0.116), followed by drainage catch basin distance (0.079) and terrain slope (0.053). Bias diagnostics reveal that Staten Island has the highest error rate (27.7%) and Brooklyn the highest false-negative rate (26.2%), suggesting systematic underdetection in high-density areas.

**Intensity regression.** All regression models achieve near-zero R² on the test set (best: ~0.06 for Random Forest Regressor), indicating that complaint intensity is largely unpredictable from the available features. Census socioeconomic variables show no recoverable importance in the CSV output, likely because linear models — which dominate this task — do not produce tree-based feature importances. This null result is itself informative: it suggests that intensity variation is driven primarily by unmeasured local factors (pipe condition, subsurface drainage capacity, storm event specifics) not captured in the current feature set.

**Resolution time regression.** The Random Forest Regressor on the temporal split achieves test R² = 0.563 and MAE = 18.8 hours. The dominant predictor is maximum tidal height during the event window (importance = 0.242), followed by precipitation event count (0.106). Borough-level bias diagnostics reveal a dramatic outlier: Queens MAE = 99.1 hours versus Bronx MAE = 11.3 hours, a nearly ninefold difference that warrants targeted investigation.

**Clustering.** Eight event archetypes are identified from the combined feature space. Physical-forcing clusters achieve the highest silhouette score (0.554 for k=2), while the combined feature space yields moderate separation (silhouette = 0.124 at k=8). Isolation Forest flags approximately 2% of events as anomalous.

## V. Conclusions

This study demonstrates that NYC flood infrastructure stress can be modeled from routinely collected administrative and environmental data with meaningful predictive accuracy at two of three stages of the stress cycle.

Occurrence prediction generalizes well across administrations: the Random Forest achieves ROC-AUC ≈ 0.89 under both stratified and temporal splits, validating that structural flood risk factors — codified in FEMA zone designations and reflected in street network travel time — are stable, temporally persistent signals. This stability has direct operational value: an occurrence model could guide proactive drain inspection and catch basin maintenance prioritization *before* storms arrive, without requiring real-time inputs beyond publicly available tide and precipitation forecasts.

The strong role of tidal forcing in resolution time (max_tide as the dominant predictor with importance 0.242) identifies a specific operational bottleneck: when flooding coincides with high-tide windows, drainage is impaired for the full tidal cycle, and crews cannot effectively intervene until water recedes. Tide-aware scheduling of maintenance crews — deploying pre-storm in high-tide-risk areas — could materially reduce resolution times in tide-exposed neighborhoods such as Red Hook, Howard Beach, and Gerritsen Beach.

The equity analysis reveals spatially heterogeneous prediction accuracy that correlates loosely with borough-level socioeconomic characteristics. Queens' outsized resolution-time residuals (MAE 99 hours vs. 11 for the Bronx) warrant dedicated follow-up: they may reflect systematic differences in complaint closure protocols, systematic under-reporting of actual physical resolution, or genuinely more complex flood dynamics in high-complaint-volume corridors. These findings reinforce calls for neighborhood-specific service level agreements rather than uniform city-wide targets, and they suggest that current infrastructure response planning may inadvertently encode geographic inequity.

Future work should integrate real-time drainage sensor readings, incorporate explicit tidal phase as a feature, and pursue causal identification strategies — exploiting capital project timing or administration transitions — to disentangle structural from behavioral drivers of flood response latency.

## VI. Limitations

Several limitations constrain the generalizability and causal interpretation of these findings.

**Reporting bias.** The 311 data source reflects citizen willingness and capacity to report, not the objective spatial distribution of flood occurrence. Low-reporting communities — often immigrant, elderly, linguistically isolated, or skeptical of government response — may generate systematically fewer complaints per flood event. This leads the occurrence model to conflate reporting propensity with physical flood risk, potentially underestimating stress in neighborhoods where civic engagement is lower. The intensity measure (complaint count per event window) is especially vulnerable to this conflation.

**Spatial resolution mismatch.** Socioeconomic features from the American Community Survey are available at the census tract level, which can span multiple blocks and mask fine-grained variation. Block-group or parcel-level indicators, when available, would likely improve the explanatory power of socioeconomic predictors.

**Temporal confounding.** The forward-looking temporal split by administration creates an inherent confound: Adams/Mamdani-era test data reflects both a change in political administration and a shift in climate dynamics, infrastructure investment levels, and data collection practices. The model cannot cleanly distinguish political learning effects from non-stationarity in flood patterns.

**Intensity model failure.** The near-zero R² for intensity regression suggests that critical drivers of complaint volume variation — pipe condition, subsurface geology, local micro-topography, block-level imperviousness — are absent from the current feature set. Without these, the intensity model has limited practical utility.

**No causal identification.** This study is entirely observational. The associations between predictors and outcomes could reflect confounding by unmeasured variables. Quasi-experimental designs exploiting infrastructure investment timing, storm intensity thresholds, or administration transitions are a natural next step for causal inference.

**Scope.** The analysis covers reported street-level flooding only. Basement flooding, subway disruptions, and building-interior flooding generate separate data streams that are not integrated here, potentially underestimating total flood stress in high-density residential corridors.